# 01. Data selection (Chapter 3)

This notebook loads the two raw Djinni datasets, describes their structure and selects the IT part that is used in the thesis.

- **Input:** `data/raw/djinni_candidates/`, `data/raw/djinni_jobs/` (Parquet files, downloaded from Hugging Face, not modified).
- **Output:** `data/processed/candidates_it_v1.csv`, `data/processed/jobs_it_v1.csv` and small preview files in `data/samples/`.

Sources:
- Candidates: https://huggingface.co/datasets/lang-uk/recruitment-dataset-candidate-profiles-english (MIT)
- Jobs: https://huggingface.co/datasets/lang-uk/recruitment-dataset-job-descriptions-english (MIT)
- Paper: N. Drushchak, M. Romanyshyn, *Introducing the Djinni Recruitment Dataset: A Corpus of Anonymized CVs and Job Postings*, UNLP 2024.

Text cleaning, EDA and normalization are done in notebook 02 (Chapter 4). Here only the selection of rows is done.

In [16]:
from pathlib import Path

import pandas as pd

ROOT = Path("..")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
SAMPLES = ROOT / "data" / "samples"
PROCESSED.mkdir(parents=True, exist_ok=True)
SAMPLES.mkdir(parents=True, exist_ok=True)

SEED = 5267
JOB_SAMPLE_SIZE = 20_000

pd.set_option("display.max_colwidth", 80)
print("pandas", pd.__version__)

pandas 3.1.0


## 1. Load raw data

In [17]:
candidates_raw = pd.read_parquet(RAW / "djinni_candidates" / "train-00000-of-00001.parquet")
jobs_raw = pd.read_parquet(RAW / "djinni_jobs" / "train-00000-of-00001.parquet")

print("candidates:", candidates_raw.shape)
print("jobs:      ", jobs_raw.shape)

candidates: (210250, 11)
jobs:       (141897, 10)


## 2. Structure of the raw files

Column names, types, number of missing values and one example value. This is the basis for the codebook in Chapter 3.

In [18]:
def describe_columns(df):
    first = df.iloc[0]
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_in_%": (df.isna().mean() * 100).round(1),
        "unique": df.nunique(),
        "example": [str(first[c])[:60].replace("\n", " ") for c in df.columns],
    })


describe_columns(candidates_raw)

,dtype,missing,missing_in_%,unique,example
Position,str,0,0.0,52913,"13 years of exp || Solidity, C#, JavaScript || CTO / VP / Te"
Moreinfo,str,0,0.0,210156,Who am I:\r - 13 years of commercial experience as a software
Looking For,str,108195,51.5,98880,I am interested in:\r - part-time engagement;\r - blockchain p
Highlights,str,104721,49.8,105036,Landed a role of Director of Blockchain Development in a met
Primary Keyword,str,43,0.0,41,Lead
English Level,str,485,0.2,6,fluent
Experience Years,float64,0,0.0,15,11.0
CV,str,0,0.0,210250,Landed a role of Director of Blockchain Development in a met
CV_lang,str,0,0.0,1,en
id,str,0,0.0,210250,50534b61-6826-52b1-9ac5-bfd2cfa348ec


In [19]:
describe_columns(jobs_raw)

,dtype,missing,missing_in_%,unique,example
Position,str,0,0.0,69671,10 + Blockchain Nodes / Masternodes to set up
Long Description,str,0,0.0,141897,*Requirements*\r \r We're looking for a long term collaboratio
Company Name,str,0,0.0,10894,MyCointainer
Exp Years,str,0,0.0,5,2y
Primary Keyword,str,0,0.0,45,Sysadmin
English Level,str,7539,5.3,5,intermediate
Published,str,0,0.0,45,2020-10-01T00:00:00+03:00
Long Description_lang,str,0,0.0,1,en
id,str,0,0.0,141897,c0ca96e7-85df-50df-a64e-d934cd02a170
__index_level_0__,int64,0,0.0,141897,27461


Notes:
- All texts are already in English (`CV_lang` and `Long Description_lang` contain only `en`).
- `id` is unique in both files and is used as the record key.
- `__index_level_0__` is a technical index left from the original export. It has no meaning and is dropped.
- `Looking For` and `Highlights` are optional fields, about half of the candidates left them empty.

In [20]:
print(candidates_raw["CV_lang"].value_counts().to_dict())
print(jobs_raw["Long Description_lang"].value_counts().to_dict())
print("candidate ids unique:", candidates_raw["id"].is_unique)
print("job ids unique:      ", jobs_raw["id"].is_unique)

{'en': 210250}
{'en': 141897}
candidate ids unique: True
job ids unique:       True


## 3. Roles in the data

`Primary Keyword` is the main specialization chosen on the Djinni platform. It is used to select IT roles.

In [21]:
roles = pd.DataFrame({
    "candidates": candidates_raw["Primary Keyword"].value_counts(),
    "jobs": jobs_raw["Primary Keyword"].value_counts(),
}).fillna(0).astype(int).sort_values("candidates", ascending=False)
roles

,candidates,jobs
Primary Keyword,,
JavaScript,33980,17903
QA,25089,6774
Design,15068,4020
Project Manager,12499,4618
Java,11463,8712
.NET,9388,7826
Marketing,8984,6933
Python,7588,5735
PHP,6947,5740


## 4. Selection of IT roles

Only technical roles are kept. Each role is mapped to a broader role family, which is used later to build balanced evaluation samples.

Excluded: non-technical roles (Design, Marketing, HR, Recruiter, Sales, Project/Product Manager, Business Analyst, SEO, Lead Generation, Artist, Scrum Master, Technical Writing, Support, Salesforce) and unclear ones (Other, Lead, missing value). Sysadmin is excluded because it is IT operations rather than software development.

In [22]:
ROLE_FAMILY = {
    "Java": "Backend", ".NET": "Backend", "Python": "Backend", "PHP": "Backend",
    "Node.js": "Backend", "Golang": "Backend", "Ruby": "Backend", "Scala": "Backend",
    "Rust": "Backend", "C++": "Backend",
    "JavaScript": "Frontend / JavaScript",
    "iOS": "Mobile", "Android": "Mobile", "Flutter": "Mobile",
    "Unity": "GameDev",
    "Data Science": "Data / ML", "Data Engineer": "Data / ML",
    "Data Analyst": "Data / ML", "SQL": "Data / ML",
    "DevOps": "DevOps",
    "QA": "QA", "QA Automation": "QA",
    "Security": "Security",
}
IT_ROLES = list(ROLE_FAMILY)
print(len(IT_ROLES), "IT roles")

23 IT roles


In [23]:
def select_it(df):
    out = df[df["Primary Keyword"].isin(IT_ROLES)].copy()
    out["role_family"] = out["Primary Keyword"].map(ROLE_FAMILY)
    return out.drop(columns="__index_level_0__").reset_index(drop=True)


candidates_it = select_it(candidates_raw)
jobs_it_all = select_it(jobs_raw)

print("candidates:", len(candidates_raw), "->", len(candidates_it))
print("jobs:      ", len(jobs_raw), "->", len(jobs_it_all))

candidates: 210250 -> 132220
jobs:       141897 -> 95474


## 5. Job sample

The search index does not need all ~95k IT job postings. Encoding them with Sentence-BERT on a laptop CPU would take a long time and would not change the comparison between methods. A random sample of 20,000 postings is used (random_state=5267). A simple random sample keeps the role proportions of the full IT subset, which is checked in the table below.

In [24]:
jobs_it = jobs_it_all.sample(n=JOB_SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

share = pd.DataFrame({
    "all IT jobs %": (jobs_it_all["role_family"].value_counts(normalize=True) * 100).round(1),
    "sample %": (jobs_it["role_family"].value_counts(normalize=True) * 100).round(1),
})
share

,all IT jobs %,sample %
role_family,,
Backend,44.4,44.1
Frontend / JavaScript,18.8,19.2
QA,14.5,14.6
DevOps,8.4,8.4
Data / ML,6.2,6.3
Mobile,5.7,5.5
GameDev,1.2,1.2
Security,0.9,0.8


## 6. Summary of the selection

In [25]:
summary = pd.DataFrame([
    {"step": "raw file", "candidates": len(candidates_raw), "jobs": len(jobs_raw)},
    {"step": "IT roles only", "candidates": len(candidates_it), "jobs": len(jobs_it_all)},
    {"step": "random job sample", "candidates": len(candidates_it), "jobs": len(jobs_it)},
])
summary

,step,candidates,jobs
0,raw file,210250,141897
1,IT roles only,132220,95474
2,random job sample,132220,20000


In [26]:
families = pd.DataFrame({
    "candidates": candidates_it["role_family"].value_counts(),
    "jobs (sample)": jobs_it["role_family"].value_counts(),
})
families

,candidates,jobs (sample)
role_family,,
Backend,46510,8826
Data / ML,6764,1268
DevOps,4120,1671
Frontend / JavaScript,33980,3831
GameDev,1815,247
Mobile,7405,1091
QA,30845,2910
Security,781,156


## 7. Save results

- `data/processed/*_v1.csv` — selected rows, original text not changed. Notebook 02 reads these files.
- `data/samples/*_sample.csv` — 50 rows of each file, so the data can be viewed directly on GitHub.

In [27]:
candidates_it.to_csv(PROCESSED / "candidates_it_v1.csv", index=False, encoding="utf-8")
jobs_it.to_csv(PROCESSED / "jobs_it_v1.csv", index=False, encoding="utf-8")

candidates_it.sample(50, random_state=SEED).to_csv(SAMPLES / "djinni_candidates_sample.csv", index=False, encoding="utf-8")
jobs_it.sample(50, random_state=SEED).to_csv(SAMPLES / "djinni_jobs_sample.csv", index=False, encoding="utf-8")

for path in sorted(PROCESSED.glob("*.csv")) + sorted(SAMPLES.glob("*.csv")):
    print(f"{path.relative_to(ROOT)}  {path.stat().st_size / 1e6:.1f} MB")

data\processed\candidates_it_v1.csv  247.2 MB
data\processed\jobs_it_v1.csv  37.3 MB
data\samples\djinni_candidates_sample.csv  0.1 MB
data\samples\djinni_jobs_sample.csv  0.1 MB
